# PDF dataset stats + sampled timing report

Scans a folder of PDFs (Phase 1 only -- no OCR, no rasterization, no P2/P3 backends) to collect per-page width/height and native-text/vector counts, visualizes their distribution, picks 10 representative `(pdf, page)` samples by two complexity metrics, runs `generate_pipeline_report.py` on just those samples, and plots pipeline time against each metric.

All outputs land under `outputs/pdf_dataset_stats/<run_tag>/`.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd() / 'pdf_dataset_stats.ipynb').exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import csv
import json
from datetime import datetime

import matplotlib.pyplot as plt

from rastervec.commons.logging_setup import configure_logging, get_logger
from rastervec.commons.paths import output_dir
from rastervec.P1_Reading_Native.reader import Reader
from rastervec.P1_Reading_Native.native_text import extract_native_text
from rastervec.P1_Reading_Native.vector_extract import extract_vectors
from rastervec.core.parallel.pool import warmup
from rastervec.Evaluation import dump_io
from rastervec.Evaluation.Evaluate.timing import flatten_page_timing
from scripts.report_config import ReportConfig
from scripts import generate_pipeline_report

configure_logging()
_LOG = get_logger('pdf_dataset_stats')

## Parameters

In [ ]:
# Folder of PDFs to scan (searched recursively for *.pdf).
INPUT_FOLDER = PROJECT_ROOT / 'references'

# An existing ReportConfig JSON used as a template for pipeline/p2/p3/
# enable_fast/dpi/debug_images -- only input_files/pages/input_dir/benchmark/
# output_root are overridden below.
BASE_REPORT_CONFIG_PATH = PROJECT_ROOT / 'scripts' / 'report_configs' / 'full_current.json'

# How many pages to sample per metric (2 metrics -> up to 2 * this many report runs).
SAMPLES_PER_METRIC = 5

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = output_dir('pdf_dataset_stats', RUN_TAG)

assert Path(INPUT_FOLDER).is_dir(), f'not a folder: {INPUT_FOLDER}'
assert Path(BASE_REPORT_CONFIG_PATH).is_file(), f'missing base config: {BASE_REPORT_CONFIG_PATH}'
print('input folder:', INPUT_FOLDER)
print('base report config:', BASE_REPORT_CONFIG_PATH)
print('run dir:', RUN_DIR)

## 1. Scan the dataset (Phase 1 only)

For every PDF under `INPUT_FOLDER`, opens it via `Reader` and calls `extract_native_text`/`extract_vectors` directly on each page -- no rasterization, no OCR, no P2/P3 backends. A corrupt/unreadable PDF is logged and skipped rather than aborting the whole scan.

In [ ]:
def scan_pdf(pdf_path: Path) -> list[dict]:
    # One record per page: cheap Phase-1-only stats (no rendering/OCR).
    records = []
    with Reader(str(pdf_path)) as reader:
        for page in reader.iter_pages():
            n_text = len(extract_native_text(page))
            n_vec = len(extract_vectors(page))
            records.append({
                'pdf_path': str(pdf_path),
                'page_index': page.meta.index,
                'width': page.meta.width,
                'height': page.meta.height,
                'num_native_text': n_text,
                'num_vectors': n_vec,
            })
    return records


pdf_paths = sorted(Path(INPUT_FOLDER).rglob('*.pdf'))
print(f'found {len(pdf_paths)} PDF(s) under {INPUT_FOLDER}')

page_records: list[dict] = []
skipped: list[tuple[Path, str]] = []
for i, pdf_path in enumerate(pdf_paths, 1):
    try:
        page_records.extend(scan_pdf(pdf_path))
    except Exception as exc:  # keep scanning the rest of the dataset
        skipped.append((pdf_path, str(exc)))
        _LOG.warning('skipping %s: %s', pdf_path, exc)
    if i % 25 == 0 or i == len(pdf_paths):
        print(f'  scanned {i}/{len(pdf_paths)} PDF(s), {len(page_records)} page(s) so far')

print(f'done: {len(page_records)} page(s) across {len(pdf_paths) - len(skipped)} PDF(s)'
      + (f', {len(skipped)} skipped' if skipped else ''))

stats_path = RUN_DIR / 'page_stats.txt'
with open(stats_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'pdf_path', 'page_index', 'width', 'height', 'num_native_text', 'num_vectors',
    ])
    writer.writeheader()
    writer.writerows(page_records)
print('wrote', stats_path)

if skipped:
    skipped_path = RUN_DIR / 'skipped_pdfs.txt'
    with open(skipped_path, 'w', encoding='utf-8') as f:
        for path, err in skipped:
            print(f'{path} - {err}', file=f)
    print('wrote', skipped_path)

## 2. Histograms

In [ ]:
def size_of(r: dict) -> float:
    return r['width'] * r['height']


def complexity_of(r: dict) -> float:
    return r['num_native_text'] + r['num_vectors']


HIST_METRICS = [
    ('width', lambda r: r['width'], 'width (pt)'),
    ('height', lambda r: r['height'], 'height (pt)'),
    ('width_x_height', size_of, 'width * height (pt^2)'),
    ('num_native_text', lambda r: r['num_native_text'], 'native text items'),
    ('num_vectors', lambda r: r['num_vectors'], 'vectors'),
    ('num_native_text_plus_vectors', complexity_of, 'native text + vectors'),
]

hist_dir = RUN_DIR / 'histograms'
hist_dir.mkdir(parents=True, exist_ok=True)

for name, fn, label in HIST_METRICS:
    values = [fn(r) for r in page_records]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(values, bins=30, color='#2563eb', edgecolor='white')
    ax.set_xlabel(label)
    ax.set_ylabel('page count')
    ax.set_title(f'Distribution of {label} ({len(values)} pages)')
    fig.tight_layout()
    fig.savefig(hist_dir / f'{name}.png', dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)

print('wrote histograms to', hist_dir)

## 3. Rank-based, forced-unique sampling

5 pages evenly spaced by rank (0th/25th/50th/75th/100th percentile) of `width * height`, then 5 more evenly spaced by rank of `num_native_text + num_vectors` -- skipping any page already picked by the first group and searching outward from the target rank on a collision, guaranteeing 10 distinct `(pdf, page)` pairs (fewer only if the dataset itself has under 10 pages).

In [ ]:
def rank_sample(records: list[dict], metric_fn, n: int, used: set[tuple[str, int]]) -> list[dict]:
    # Evenly spaced by rank/percentile over `records` sorted by `metric_fn`,
    # skipping any (pdf_path, page_index) already in `used`, searching
    # outward from the target rank when a collision occurs. Mutates `used`.
    ordered = sorted(records, key=metric_fn)
    picked = []
    n_avail = len(ordered)
    if n_avail == 0:
        return picked
    fracs = [i / (n - 1) if n > 1 else 0.0 for i in range(n)]
    for frac in fracs:
        target = round(frac * (n_avail - 1))
        chosen_idx = None
        for offset in range(n_avail):
            for idx in (target - offset, target + offset):
                if 0 <= idx < n_avail:
                    key = (ordered[idx]['pdf_path'], ordered[idx]['page_index'])
                    if key not in used:
                        chosen_idx = idx
                        break
            if chosen_idx is not None:
                break
        if chosen_idx is None:
            continue  # dataset exhausted
        rec = ordered[chosen_idx]
        used.add((rec['pdf_path'], rec['page_index']))
        picked.append(rec)
    return picked


used_pages: set[tuple[str, int]] = set()
size_samples = rank_sample(page_records, size_of, SAMPLES_PER_METRIC, used_pages)
for r in size_samples:
    r['criterion'] = 'width_x_height'
    r['metric_value'] = size_of(r)

complexity_samples = rank_sample(page_records, complexity_of, SAMPLES_PER_METRIC, used_pages)
for r in complexity_samples:
    r['criterion'] = 'native_text_plus_vectors'
    r['metric_value'] = complexity_of(r)

samples = size_samples + complexity_samples
print(f'sampled {len(samples)} distinct (pdf, page) pairs '
      f'({len(size_samples)} by size, {len(complexity_samples)} by complexity)')
for r in samples:
    crit = r['criterion']
    name = Path(r['pdf_path']).name
    idx = r['page_index']
    val = r['metric_value']
    print(f'  [{crit}] {name} page {idx} (value={val:.0f})')

sample_path = RUN_DIR / 'sample_pages.json'
sample_path.write_text(json.dumps(samples, indent=2), encoding='utf-8')
print('wrote', sample_path)

## 4. Build and run the pipeline report on the sample

Uses `BASE_REPORT_CONFIG_PATH` as a template for `pipeline`/`p2`/`p3`/`enable_fast`/`dpi`/`debug_images`, overriding only `input_files`/`pages`/`output_root` to target exactly the 10 sampled pages and forcing `benchmark: false` (its GT-scoring artifacts are unrelated to timing and would just add cost). `warmup()` runs first so the first sampled page's timing is not skewed by one-off model loading.

In [ ]:
base_config_dict = json.loads(Path(BASE_REPORT_CONFIG_PATH).read_text(encoding='utf-8'))

pages_by_stem: dict[str, list[int]] = {}
for r in samples:
    stem = Path(r['pdf_path']).stem
    pages_by_stem.setdefault(stem, [])
    if r['page_index'] not in pages_by_stem[stem]:
        pages_by_stem[stem].append(r['page_index'])

merged_config_dict = dict(base_config_dict)
merged_config_dict.pop('input_dir', None)
merged_config_dict['input_files'] = sorted({r['pdf_path'] for r in samples})
merged_config_dict['pages'] = pages_by_stem
merged_config_dict['benchmark'] = False
merged_config_dict['output_root'] = str(RUN_DIR / 'pipeline_reports')

report_config = ReportConfig(**merged_config_dict)
generated_config_path = RUN_DIR / 'generated_report_config.json'
generated_config_path.write_text(report_config.model_dump_json(indent=2), encoding='utf-8')

num_pdfs = len(merged_config_dict['input_files'])
print('wrote', generated_config_path)
print(f'running generate_pipeline_report on {num_pdfs} PDF(s), {len(samples)} sampled page(s)')

warmup()  # avoid a cold-start model-load skewing the first sample's timing
exit_code = generate_pipeline_report.main(['--config', str(generated_config_path)])
assert exit_code == 0, f'generate_pipeline_report failed (exit {exit_code})'

## 5. Extract timing and plot

Timing comes from each sample's own `dump.json` -- `step_durations`/`substep_durations`/`debug_durations`, written by `generate_pipeline_report.py` regardless of the `benchmark` flag -- via `Evaluation.Evaluate.timing.flatten_page_timing`, not a hand-rolled wall-clock wrapper.

In [ ]:
report_output_root = Path(merged_config_dict['output_root'])
candidate_dirs = sorted(report_output_root.glob(f'*__{generated_config_path.stem}'))
assert candidate_dirs, f'no report run folder found under {report_output_root}'
report_run_dir = candidate_dirs[-1]
print('report run dir:', report_run_dir)

for r in samples:
    stem = Path(r['pdf_path']).stem
    page_index = r['page_index']
    dump_path = report_run_dir / stem / 'dump.json'
    r['timing'] = None
    if not dump_path.is_file():
        _LOG.warning('no dump.json for %s (page %d)', stem, page_index)
        continue
    dump = dump_io.load_dump(dump_path)
    page_dump = next((p for p in dump.pages if p.page_meta.index == page_index), None)
    if page_dump is None:
        _LOG.warning('dump.json for %s has no page %d', stem, page_index)
        continue
    timing = flatten_page_timing(
        page_dump.step_durations, page_dump.substep_durations, page_dump.debug_durations,
    )
    if not timing:
        _LOG.warning('no timing recorded for %s page %d', stem, page_index)
        continue
    r['timing'] = timing

timed_samples = [r for r in samples if r['timing']]
print(f'{len(timed_samples)}/{len(samples)} sample(s) have timing data')

In [ ]:
TIMING_METRICS = [
    ('width_x_height', size_of, 'width * height (pt^2)'),
    ('num_native_text', lambda r: r['num_native_text'], 'native text items'),
    ('num_vectors', lambda r: r['num_vectors'], 'vectors'),
    ('num_native_text_plus_vectors', complexity_of, 'native text + vectors'),
]

timing_plot_dir = RUN_DIR / 'timing_plots'
timing_plot_dir.mkdir(parents=True, exist_ok=True)

for name, fn, label in TIMING_METRICS:
    xs = [fn(r) for r in timed_samples]
    ys = [r['timing']['total'] for r in timed_samples]
    colors = ['#2563eb' if r['criterion'] == 'width_x_height' else '#f97316' for r in timed_samples]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(xs, ys, c=colors, s=60, edgecolor='white')
    ax.set_xlabel(label)
    ax.set_ylabel('pipeline time (s, phase1-4 total)')
    ax.set_title(f'Pipeline time vs {label}')
    fig.tight_layout()
    fig.savefig(timing_plot_dir / f'time_vs_{name}.png', dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)

print('wrote timing scatter plots to', timing_plot_dir)

In [ ]:
phase_keys = ['phase1', 'phase2', 'phase3', 'phase4']
phase_colors = {'phase1': '#2563eb', 'phase2': '#16a34a', 'phase3': '#f97316', 'phase4': '#a855f7'}

ordered_timed = sorted(timed_samples, key=lambda r: r['metric_value'])
labels = []
for r in ordered_timed:
    stem = Path(r['pdf_path']).stem[:12]
    page_index = r['page_index']
    labels.append(f'{stem} p{page_index}')

fig, ax = plt.subplots(figsize=(max(6, len(ordered_timed) * 1.2), 5))
bottoms = [0.0] * len(ordered_timed)
for phase in phase_keys:
    values = [r['timing'].get(phase, 0.0) for r in ordered_timed]
    ax.bar(labels, values, bottom=bottoms, label=phase, color=phase_colors[phase])
    bottoms = [b + v for b, v in zip(bottoms, values)]
ax.set_ylabel('seconds')
ax.set_title('Per-phase pipeline time per sampled page')
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
fig.tight_layout()
fig.savefig(timing_plot_dir / 'phase_breakdown.png', dpi=120, bbox_inches='tight')
plt.show()
plt.close(fig)

print('wrote', timing_plot_dir / 'phase_breakdown.png')

In [ ]:
results_path = RUN_DIR / 'timing_results.txt'
with open(results_path, 'w', encoding='utf-8') as f:
    for r in samples:
        pdf_path = r['pdf_path']
        page_index = r['page_index']
        criterion = r['criterion']
        metric_value = r['metric_value']
        width = r['width']
        height = r['height']
        num_native_text = r['num_native_text']
        num_vectors = r['num_vectors']
        print(f'pdf: {pdf_path}', file=f)
        print(f'  page_index: {page_index}', file=f)
        print(f'  criterion: {criterion} (value={metric_value:.0f})', file=f)
        print(f'  width x height: {width:.1f} x {height:.1f}', file=f)
        print(f'  num_native_text: {num_native_text}, num_vectors: {num_vectors}', file=f)
        if r['timing']:
            print('  timing:', file=f)
            for k, v in r['timing'].items():
                print(f'    {k}: {v:.4f}s', file=f)
        else:
            print('  timing: MISSING', file=f)
        print(file=f)
print('wrote', results_path)